# Liquid Clustering vs. Partitioning — BookMyShow bookings

Build a ~1 GB partitioned Delta table, benchmark it, migrate to liquid clustering,
benchmark again — and check whether each step actually rewrites data or not.

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
table_name = dbutils.widgets.get("table_name")
num_rows = int(dbutils.widgets.get("num_rows"))
full_table_name = f"{catalog}.{schema}.{table_name}"

print(f"Target table: {full_table_name}")
print(f"Row count to generate: {num_rows:,}")

## 1. Generate a ~1 GB synthetic bookings dataset

10 dates x 10 hours. Tune the `num_rows` widget if the printed size lands far from 1 GB.

In [0]:
import time
from pyspark.sql import functions as F

FIRST_NAMES = ["Aditi", "Rohan", "Priya", "Karan", "Neha", "Arjun", "Isha", "Vikram", "Sneha", "Rahul",
               "Ananya", "Aarav", "Divya", "Kabir", "Meera", "Sanjay", "Pooja", "Rajesh", "Anjali", "Varun",
               "Simran", "Nikhil", "Kavya", "Aman", "Riya", "Siddharth", "Tanvi", "Yash", "Ishita", "Manish"]
LAST_NAMES = ["Sharma", "Verma", "Patel", "Gupta", "Reddy", "Nair", "Iyer", "Menon", "Kapoor", "Malhotra",
              "Chopra", "Bose", "Rao", "Desai", "Joshi", "Mehta", "Kulkarni", "Pillai", "Shah", "Bansal"]
MOVIES = ["Pathaan", "Jawan", "Animal", "Stree 2", "Kalki 2898 AD", "Fighter", "Rocky Aur Rani",
          "Gadar 2", "OMG 2", "Dunki", "Sam Bahadur", "Tiger 3", "Bhool Bhulaiyaa 3",
          "Singham Again", "Munjya"]
THEATRES = ["PVR Phoenix", "INOX Nariman Point", "Cinepolis Andheri", "PVR Juhu", "INOX R City",
            "Cinepolis Fun Republic", "PVR ICON", "INOX Megaplex", "Carnival IMAX", "Miraj Cinemas"]
CITIES = ["Mumbai", "Delhi", "Bengaluru", "Hyderabad", "Chennai", "Pune", "Kolkata", "Ahmedabad"]
PAYMENT_STATUS = ["SUCCESS", "SUCCESS", "SUCCESS", "SUCCESS", "REFUNDED", "FAILED"]  # skewed toward SUCCESS

BOOKING_DATES = [f"2026-07-{d:02d}" for d in range(1, 11)]  # 10 distinct partition dates
BOOKING_HOURS = [9, 10, 11, 12, 13, 14, 18, 19, 20, 21]     # 10 distinct partition hours

def pick(values):
    arr = F.array([F.lit(v) for v in values])
    idx = (F.rand() * len(values)).cast("int")
    return F.element_at(arr, idx + 1)

In [0]:
bookings_df = (
    spark.range(0, num_rows)
    .withColumnRenamed("id", "booking_id")
    .withColumn("booking_date", pick(BOOKING_DATES).cast("date"))
    .withColumn("booking_hour", pick(BOOKING_HOURS).cast("int"))
    .withColumn("user_first_name", pick(FIRST_NAMES))
    .withColumn("user_last_name", pick(LAST_NAMES))
    .withColumn("movie_name", pick(MOVIES))
    .withColumn("theatre_name", pick(THEATRES))
    .withColumn("city", pick(CITIES))
    .withColumn("payment_status", pick(PAYMENT_STATUS))
    .withColumn("seat_count", (F.rand() * 5 + 1).cast("int"))
    .withColumn("amount", F.round(F.rand() * 4500 + 150, 2))
    .withColumn(
        "booking_number",
        F.concat(
            F.lit("BMS"),
            F.date_format("booking_date", "yyyyMMdd"),
            F.lit("-"),
            F.lpad(F.col("booking_id").cast("string"), 10, "0"),
        ),
    )
    .withColumn(
        "booking_timestamp",
        F.to_timestamp(
            F.concat_ws(" ", F.col("booking_date").cast("string"),
                        F.format_string("%02d:00:00", F.col("booking_hour")))
        ),
    )
    .select(
        "booking_id", "booking_number", "user_first_name", "user_last_name",
        "movie_name", "theatre_name", "city", "seat_count", "amount",
        "payment_status", "booking_timestamp", "booking_date", "booking_hour",
    )
)

## 2. Write the managed, partitioned Delta table

Repartitioned into 400 chunks before writing to deliberately create many small
files per partition — the small-file problem clustering + `OPTIMIZE` fixes.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")

NUM_WRITE_PARTITIONS = 400  # >> 100 (10 dates x 10 hours) on purpose, to fan out into many small files

t0 = time.time()
(
    bookings_df
    .repartition(NUM_WRITE_PARTITIONS)
    .write.format("delta")
    .mode("overwrite")
    .partitionBy("booking_date", "booking_hour")
    .saveAsTable(full_table_name)
)
write_seconds = time.time() - t0
print(f"Initial partitioned write took {write_seconds:.1f}s")

In [0]:
detail_initial = spark.sql(f"DESCRIBE DETAIL {full_table_name}").collect()[0]
size_gb = detail_initial["sizeInBytes"] / (1024 ** 3)
print(f"Table size: {size_gb:.2f} GB across {detail_initial['numFiles']} files")
print("If size_gb is well below 1.0 (or you want it larger), bump the num_rows widget and rerun "
      "the generation + write cells above.")

## 3. Baseline query performance (before clustering)

Neither query filters on the partition columns, so both are full-table scans.

In [0]:
def time_query(sql_text, runs=3):
    # Note: spark.catalog.clearCache() / CLEAR CACHE is not supported on serverless
    # compute, so these timings reflect whatever caching serverless does internally.
    durations = []
    row_count = None
    for _ in range(runs):
        t0 = time.time()
        result = spark.sql(sql_text).collect()
        durations.append(time.time() - t0)
        row_count = len(result)
    return {"sql": sql_text, "runs": durations, "best": min(durations),
            "avg": sum(durations) / len(durations), "rows": row_count}

sample_first_name = spark.sql(f"SELECT user_first_name FROM {full_table_name} LIMIT 1").first()[0]
sample_booking_number = spark.sql(f"SELECT booking_number FROM {full_table_name} LIMIT 1").first()[0]

name_query = f"SELECT * FROM {full_table_name} WHERE user_first_name = '{sample_first_name}'"
booking_query = f"SELECT * FROM {full_table_name} WHERE booking_number = '{sample_booking_number}'"

print(f"Sampling for: user_first_name = {sample_first_name!r}, booking_number = {sample_booking_number!r}")

In [0]:
baseline_name_result = time_query(name_query)
baseline_booking_result = time_query(booking_query)

print(f"[BEFORE clustering] user_first_name search    -> best {baseline_name_result['best']:.2f}s, "
      f"avg {baseline_name_result['avg']:.2f}s, rows={baseline_name_result['rows']:,}")
print(f"[BEFORE clustering] booking_number search      -> best {baseline_booking_result['best']:.2f}s, "
      f"avg {baseline_booking_result['avg']:.2f}s, rows={baseline_booking_result['rows']:,}")

### Table layout before clustering (baseline)

In [0]:
files_before = spark.sql(f"DESCRIBE DETAIL {full_table_name}").collect()[0]
print(f"BEFORE clustering/optimize -> numFiles={files_before['numFiles']}, "
      f"sizeInBytes={files_before['sizeInBytes']:,}, "
      f"clusteringColumns={files_before['clusteringColumns']}, "
      f"partitionColumns={files_before['partitionColumns']}")

display(spark.sql(f"DESCRIBE HISTORY {full_table_name}").select("version", "timestamp", "operation", "operationMetrics"))

## 4. Try `ALTER TABLE ... CLUSTER BY` on the partitioned table

Metadata-only ALTER only works on tables that were never partitioned. Confirming that.

In [0]:
alter_error = None
t0 = time.time()
try:
    spark.sql(f"ALTER TABLE {full_table_name} CLUSTER BY (booking_date, booking_hour)")
    alter_seconds = time.time() - t0
    print(f"ALTER TABLE ... CLUSTER BY unexpectedly succeeded in {alter_seconds:.2f}s")
except Exception as e:
    alter_seconds = time.time() - t0
    alter_error = str(e).splitlines()[0]
    print(f"ALTER TABLE ... CLUSTER BY was rejected after {alter_seconds:.2f}s, as expected:")
    print(alter_error)

detail_after_alter_attempt = spark.sql(f"DESCRIBE DETAIL {full_table_name}").collect()[0]
print(f"numFiles={detail_after_alter_attempt['numFiles']}, "
      f"sizeInBytes={detail_after_alter_attempt['sizeInBytes']:,} "
      "-- unchanged vs. the BEFORE numbers, since the rejected ALTER touched nothing.")

## 5. Migrate to liquid clustering — `CREATE OR REPLACE TABLE ... CLUSTER BY ... AS SELECT`

The documented way to convert a partitioned table. Always a full rewrite.

In [0]:
t0 = time.time()
spark.sql(f"""
    CREATE OR REPLACE TABLE {full_table_name}
    CLUSTER BY (booking_date, booking_hour)
    AS SELECT * FROM {full_table_name}
""")
migrate_seconds = time.time() - t0
print(f"CREATE OR REPLACE TABLE ... CLUSTER BY ... AS SELECT took {migrate_seconds:.1f}s")

detail_after_migration = spark.sql(f"DESCRIBE DETAIL {full_table_name}").collect()[0]
print(f"AFTER migration -> numFiles={detail_after_migration['numFiles']}, "
      f"sizeInBytes={detail_after_migration['sizeInBytes']:,}, "
      f"clusteringColumns={detail_after_migration['clusteringColumns']}, "
      f"partitionColumns={detail_after_migration['partitionColumns']}")
print("partitionColumns is now empty and clusteringColumns lists booking_date/booking_hour. "
      "The file count/size reflect a brand new physical layout -- this step is a full "
      "rewrite, not a metadata-only change.")

display(spark.sql(f"DESCRIBE HISTORY {full_table_name}").select("version", "timestamp", "operation", "operationMetrics").limit(3))

## 6. Compact the freshly clustered table — `OPTIMIZE`

The maintenance command you'd run repeatedly as new data lands.

In [0]:
t0 = time.time()
optimize_result = spark.sql(f"OPTIMIZE {full_table_name}")
optimize_seconds = time.time() - t0
display(optimize_result)
print(f"OPTIMIZE took {optimize_seconds:.1f}s")

metrics_row = optimize_result.select("metrics.*").first()
print(f"numFilesAdded={metrics_row['numFilesAdded']}, numFilesRemoved={metrics_row['numFilesRemoved']}, "
      f"totalSizeAdded={metrics_row['filesAdded']['totalSize']:,}, "
      f"totalSizeRemoved={metrics_row['filesRemoved']['totalSize']:,}")
print("numFilesRemoved > 0 confirms OPTIMIZE physically rewrote the data into new, clustered files "
      "-- unlike the metadata-only ALTER TABLE above.")

### Table layout after OPTIMIZE

In [0]:
detail_after_optimize = spark.sql(f"DESCRIBE DETAIL {full_table_name}").collect()[0]
print(f"AFTER optimize -> numFiles={detail_after_optimize['numFiles']}, "
      f"sizeInBytes={detail_after_optimize['sizeInBytes']:,}")

display(spark.sql(f"DESCRIBE HISTORY {full_table_name}").select("version", "timestamp", "operation", "operationMetrics").limit(5))

## 7. Re-run the same queries after clustering + OPTIMIZE

Same values, same methodology as the baseline run.

In [0]:
clustered_name_result = time_query(name_query)
clustered_booking_result = time_query(booking_query)

print(f"[AFTER clustering+optimize] user_first_name search -> best {clustered_name_result['best']:.2f}s, "
      f"avg {clustered_name_result['avg']:.2f}s, rows={clustered_name_result['rows']:,}")
print(f"[AFTER clustering+optimize] booking_number search   -> best {clustered_booking_result['best']:.2f}s, "
      f"avg {clustered_booking_result['avg']:.2f}s, rows={clustered_booking_result['rows']:,}")

## 8. Incremental OPTIMIZE — append a small batch of new bookings

Append ~0.5% new rows, then run `OPTIMIZE` again: does it touch everything again,
or just the new data?

In [0]:
append_rows = int(dbutils.widgets.get("append_rows"))

new_bookings_df = (
    spark.range(0, append_rows)
    .withColumn("booking_id", F.col("id") + F.lit(num_rows))
    .drop("id")
    .withColumn("booking_date", pick(BOOKING_DATES).cast("date"))
    .withColumn("booking_hour", pick(BOOKING_HOURS).cast("int"))
    .withColumn("user_first_name", pick(FIRST_NAMES))
    .withColumn("user_last_name", pick(LAST_NAMES))
    .withColumn("movie_name", pick(MOVIES))
    .withColumn("theatre_name", pick(THEATRES))
    .withColumn("city", pick(CITIES))
    .withColumn("payment_status", pick(PAYMENT_STATUS))
    .withColumn("seat_count", (F.rand() * 5 + 1).cast("int"))
    .withColumn("amount", F.round(F.rand() * 4500 + 150, 2))
    .withColumn(
        "booking_number",
        F.concat(
            F.lit("BMS"),
            F.date_format("booking_date", "yyyyMMdd"),
            F.lit("-"),
            F.lpad(F.col("booking_id").cast("string"), 10, "0"),
        ),
    )
    .withColumn(
        "booking_timestamp",
        F.to_timestamp(
            F.concat_ws(" ", F.col("booking_date").cast("string"),
                        F.format_string("%02d:00:00", F.col("booking_hour")))
        ),
    )
    .select(
        "booking_id", "booking_number", "user_first_name", "user_last_name",
        "movie_name", "theatre_name", "city", "seat_count", "amount",
        "payment_status", "booking_timestamp", "booking_date", "booking_hour",
    )
)

detail_before_append = spark.sql(f"DESCRIBE DETAIL {full_table_name}").collect()[0]

t0 = time.time()
new_bookings_df.write.format("delta").mode("append").saveAsTable(full_table_name)
append_seconds = time.time() - t0

detail_after_append = spark.sql(f"DESCRIBE DETAIL {full_table_name}").collect()[0]
print(f"Appended {append_rows:,} rows in {append_seconds:.1f}s")
print(f"Before append -> numFiles={detail_before_append['numFiles']}, sizeInBytes={detail_before_append['sizeInBytes']:,}")
print(f"After append  -> numFiles={detail_after_append['numFiles']}, sizeInBytes={detail_after_append['sizeInBytes']:,}")

In [0]:
t0 = time.time()
optimize2_result = spark.sql(f"OPTIMIZE {full_table_name}")
optimize2_seconds = time.time() - t0
display(optimize2_result)
print(f"Second OPTIMIZE (after small append) took {optimize2_seconds:.1f}s")

metrics2_row = optimize2_result.select("metrics.*").first()
print(f"numFilesAdded={metrics2_row['numFilesAdded']}, numFilesRemoved={metrics2_row['numFilesRemoved']}, "
      f"totalSizeAdded={metrics2_row['filesAdded']['totalSize']:,}, "
      f"totalSizeRemoved={metrics2_row['filesRemoved']['totalSize']:,}")

detail_after_optimize2 = spark.sql(f"DESCRIBE DETAIL {full_table_name}").collect()[0]
print(f"AFTER second optimize -> numFiles={detail_after_optimize2['numFiles']}, "
      f"sizeInBytes={detail_after_optimize2['sizeInBytes']:,}")

print()
print("First OPTIMIZE (right after CTAS, nothing clustered yet):  "
      f"touched {metrics_row['numFilesRemoved']} of {detail_after_migration['numFiles']} files "
      f"({100 * metrics_row['numFilesRemoved'] / detail_after_migration['numFiles']:.0f}%) in {optimize_seconds:.1f}s")
print("Second OPTIMIZE (after a small incremental append):        "
      f"touched {metrics2_row['numFilesRemoved']} of {detail_after_append['numFiles']} files "
      f"({100 * metrics2_row['numFilesRemoved'] / detail_after_append['numFiles']:.0f}%) in {optimize2_seconds:.1f}s")
print("A much smaller fraction touched on the second run is the concrete evidence that "
      "liquid clustering re-clusters incrementally rather than rewriting the whole table "
      "every time -- unlike ZORDER, which always requires a full rewrite.")

### Note: Auto Optimize with Liquid Clustering

Liquid clustering auto-triggers eager optimization at a certain write size, so
writes often arrive pre-optimized — likely why both `OPTIMIZE` calls above showed
`numFilesAdded=0, numFilesRemoved=0`. `delta.autoOptimize.optimizeWrite = true`
(DBR 13.3+) makes this explicit; `autoCompact` has no effect on clustered tables yet.

### Note: Row-Level Concurrency on Liquid Clustered Tables

Two writers touching different rows in the same file no longer conflict —
conflicts are detected at the row level, not the file level.

- **Requires:** Deletion Vectors + an unpartitioned table.
- Automatic on liquid-clustered tables since **DBR 13.3 LTS**; GA for any
  unpartitioned table with deletion vectors since **DBR 14.3 LTS**.
- **Trade-off:** more conflict-check overhead under very high write concurrency.

## 9. Summary — performance, timing, and file layout

In [0]:
import pandas as pd

query_summary = pd.DataFrame([
    {"metric": "user_first_name search (best, s)", "before": baseline_name_result["best"], "after": clustered_name_result["best"]},
    {"metric": "user_first_name search (avg, s)",  "before": baseline_name_result["avg"],  "after": clustered_name_result["avg"]},
    {"metric": "booking_number search (best, s)",  "before": baseline_booking_result["best"], "after": clustered_booking_result["best"]},
    {"metric": "booking_number search (avg, s)",   "before": baseline_booking_result["avg"],  "after": clustered_booking_result["avg"]},
])
query_summary["improvement_%"] = (1 - query_summary["after"] / query_summary["before"]) * 100
display(query_summary)

In [0]:
layout_summary = pd.DataFrame([
    {"stage": "initial partitioned write", "numFiles": files_before["numFiles"],
     "sizeGB": files_before["sizeInBytes"] / 1024 ** 3, "op_duration_s": write_seconds},
    {"stage": "ALTER TABLE CLUSTER BY (rejected)", "numFiles": detail_after_alter_attempt["numFiles"],
     "sizeGB": detail_after_alter_attempt["sizeInBytes"] / 1024 ** 3, "op_duration_s": alter_seconds},
    {"stage": "after CREATE OR REPLACE ... CLUSTER BY", "numFiles": detail_after_migration["numFiles"],
     "sizeGB": detail_after_migration["sizeInBytes"] / 1024 ** 3, "op_duration_s": migrate_seconds},
    {"stage": "after OPTIMIZE (1st, full)", "numFiles": detail_after_optimize["numFiles"],
     "sizeGB": detail_after_optimize["sizeInBytes"] / 1024 ** 3, "op_duration_s": optimize_seconds},
    {"stage": "after incremental append", "numFiles": detail_after_append["numFiles"],
     "sizeGB": detail_after_append["sizeInBytes"] / 1024 ** 3, "op_duration_s": append_seconds},
    {"stage": "after OPTIMIZE (2nd, incremental)", "numFiles": detail_after_optimize2["numFiles"],
     "sizeGB": detail_after_optimize2["sizeInBytes"] / 1024 ** 3, "op_duration_s": optimize2_seconds},
])
display(layout_summary)

print(f"ALTER TABLE CLUSTER BY (partitioned table)      : {alter_seconds:.2f}s -- rejected, 0 files touched")
print(f"CREATE OR REPLACE ... CLUSTER BY ... AS SELECT  : {migrate_seconds:.1f}s -- full rewrite, "
      f"{files_before['numFiles']} files -> {detail_after_migration['numFiles']} files")
print(f"OPTIMIZE (1st, nothing clustered yet)           : {optimize_seconds:.1f}s -- rewrote "
      f"{metrics_row['numFilesRemoved']} of {detail_after_migration['numFiles']} files")
print(f"OPTIMIZE (2nd, after small incremental append)  : {optimize2_seconds:.1f}s -- rewrote "
      f"{metrics2_row['numFilesRemoved']} of {detail_after_append['numFiles']} files")

## Notes

- `ALTER TABLE ... CLUSTER BY` only works on tables that were never partitioned.
- Migrating a partitioned table is always a **full rewrite** — no free lunch.
- `OPTIMIZE` re-clusters incrementally afterward — only new/changed data, not everything.
- Query gains come mainly from **file compaction**, not from skipping on these columns.
- No explicit cache-clearing between runs (`CLEAR CACHE` isn't supported on
  serverless) — re-run benchmark cells if numbers look noisy.

## 10. Testing it for real — row-level concurrency conflicts

Two single-file test tables (partitioned vs. liquid-clustered), two concurrent
`UPDATE`s on different rows, one deliberately slowed down so both overlap in
time. Expect: conflict on the partitioned table, success on both on the
clustered one. Timing-dependent — re-run a cell if it doesn't match.

In [0]:
import threading
import time as _time
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql.types import LongType

CONCURRENCY_TEST_ROWS = 20000
SLOW_DELAY_PER_ROW = 0.0004  # ~8s total scan time for the "slow" UPDATE over 20k rows

partitioned_test_table = f"{full_table_name}_concurrency_partitioned"
clustered_test_table = f"{full_table_name}_concurrency_clustered"

def make_test_df():
    return (
        spark.range(0, CONCURRENCY_TEST_ROWS)
        .withColumnRenamed("id", "booking_id")
        .withColumn("dummy_partition", F.lit("ALL"))
        .withColumn("seat_count", F.lit(1))
    )

spark.sql(f"DROP TABLE IF EXISTS {partitioned_test_table}")
(
    make_test_df()
    .coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .partitionBy("dummy_partition")
    .saveAsTable(partitioned_test_table)
)

spark.sql(f"DROP TABLE IF EXISTS {clustered_test_table}")
(
    make_test_df()
    .drop("dummy_partition")
    .coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .clusterBy("booking_id")
    .saveAsTable(clustered_test_table)
)
spark.sql(f"ALTER TABLE {clustered_test_table} SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true')")

for t in (partitioned_test_table, clustered_test_table):
    d = spark.sql(f"DESCRIBE DETAIL {t}").collect()[0]
    print(f"{t}: numFiles={d['numFiles']}, partitionColumns={d['partitionColumns']}, "
          f"clusteringColumns={d['clusteringColumns']}")

In [0]:
def _slow_row_match(row_id, target_id):
    _time.sleep(SLOW_DELAY_PER_ROW)
    return row_id == target_id

spark.udf.register("slow_row_match", _slow_row_match, "boolean")

def run_concurrency_test(table_name, row_id_a=1, row_id_b=2):
    results = {}

    def slow_update():
        t0 = _time.time()
        try:
            spark.sql(
                f"UPDATE {table_name} SET seat_count = seat_count + 1 "
                f"WHERE slow_row_match(booking_id, {row_id_a})"
            )
            results["slow"] = ("SUCCESS", None, _time.time() - t0)
        except Exception as e:
            results["slow"] = ("CONFLICT", f"{type(e).__name__}: {str(e).splitlines()[0]}", _time.time() - t0)

    def fast_update():
        _time.sleep(1.0)  # let the slow UPDATE start scanning first
        t0 = _time.time()
        try:
            spark.sql(f"UPDATE {table_name} SET seat_count = seat_count + 1 WHERE booking_id = {row_id_b}")
            results["fast"] = ("SUCCESS", None, _time.time() - t0)
        except Exception as e:
            results["fast"] = ("CONFLICT", f"{type(e).__name__}: {str(e).splitlines()[0]}", _time.time() - t0)

    with ThreadPoolExecutor(max_workers=2) as pool:
        f1 = pool.submit(slow_update)
        f2 = pool.submit(fast_update)
        f1.result()
        f2.result()

    return results

### Run against the PARTITIONED table -- expect a conflict

In [0]:
partitioned_results = run_concurrency_test(partitioned_test_table)
for writer, (status, error, seconds) in partitioned_results.items():
    print(f"[{table_name} PARTITIONED] {writer:5s} update -> {status} in {seconds:.2f}s"
          + (f"  ({error})" if error else ""))

### Run against the LIQUID-CLUSTERED table -- expect both to succeed

In [0]:
clustered_results = run_concurrency_test(clustered_test_table)
for writer, (status, error, seconds) in clustered_results.items():
    print(f"[{table_name} CLUSTERED]   {writer:5s} update -> {status} in {seconds:.2f}s"
          + (f"  ({error})" if error else ""))

print()
print("If the partitioned table showed a CONFLICT and the clustered table showed "
      "SUCCESS/SUCCESS, that's row-level concurrency in action: same physical "
      "single-file layout, same two overlapping writers touching different rows -- "
      "file-level conflict detection on the partitioned table can't tell the rows "
      "were different, row-level conflict detection on the clustered table can.")

In [0]:
# Optional cleanup — uncomment to drop the demo table(s)
# spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
# spark.sql(f"DROP TABLE IF EXISTS {partitioned_test_table}")
# spark.sql(f"DROP TABLE IF EXISTS {clustered_test_table}")